# 03 - WARN PDF Extraction (Save interim CSV fragments)

Goal: Extract table-like content from CA WARN PDFs into multiple CSV fragments per PDF.

Input: data/raw/warn_pdf/*.pdf   
Output (interim): data/interim/warn_tables/*.csv

Each PDF page is extracted separately; Tabula often returns one CSV per detected table region, so a single PDF becomes many fragments.

In [8]:
# pip install tabula-py

01: Load All Extracted Tables

Goal: Load every extracted CSV fragment so we can filter out the junk tables and clean real rows.

In [97]:
import pandas as pd
import tabula
import glob
import os
import numpy as np

In [12]:
pdf_files = sorted(glob.glob("../data/raw/warn_pdf/*.pdf"))

pdf_files

['../data/raw/warn_pdf/warn7-1-20to6-3-21.pdf',
 '../data/raw/warn_pdf/warn7-1-21to6-30-22.pdf',
 '../data/raw/warn_pdf/warn7-1-22to06-30-23.pdf',
 '../data/raw/warn_pdf/warn7-1-23-to6-30-24.pdf',
 '../data/raw/warn_pdf/warn7-1-24to6-30-25.pdf']

In [13]:

os.makedirs("../data/interim/warn_tables", exist_ok=True)

for pdf in pdf_files:
    base = os.path.splitext(os.path.basename(pdf))[0]
    print(f"Extracting: {base}")
    
    dfs = tabula.read_pdf(
        pdf,
        pages="all",
        multiple_tables=True,
        stream=True,
        guess=False
    )
    print(f"  tables found: {len(dfs)}")
    
    # Save each extracted table
    for i, df in enumerate(dfs):
        out_path = f"../data/interim/warn_tables/{base}_table{i}.csv"
        df.to_csv(out_path, index=False)

Extracting: warn7-1-20to6-3-21
  tables found: 117
Extracting: warn7-1-21to6-30-22
  tables found: 31
Extracting: warn7-1-22to06-30-23
  tables found: 44
Extracting: warn7-1-23-to6-30-24
  tables found: 66
Extracting: warn7-1-24to6-30-25
  tables found: 55


02: Load extracted tables and inspect

Goal: Load all extracted CSV fragments and see what kinds of columns/headers we got.
This step helps us identify:

junk header tables (like “WARN Report” summary pages)

real data tables (notice/effective/received/company/county/employees/type)

In [18]:
table_files = sorted(glob.glob("../data/interim/warn_tables/*.csv"))
print("Total extracted CSV fragments:", len(table_files))
table_files[:5]

Total extracted CSV fragments: 313


['../data/interim/warn_tables/warn7-1-20to6-3-21_table0.csv',
 '../data/interim/warn_tables/warn7-1-20to6-3-21_table1.csv',
 '../data/interim/warn_tables/warn7-1-20to6-3-21_table10.csv',
 '../data/interim/warn_tables/warn7-1-20to6-3-21_table100.csv',
 '../data/interim/warn_tables/warn7-1-20to6-3-21_table101.csv']

In [19]:
def load_warn_fragment(file_path: str) -> pd.DataFrame:
    """
    Load one extracted WARN CSV fragment safely.
    We use header=None so pandas does not accidentally treat a data row as the header.
    """
    df = pd.read_csv(file_path, header=None)
    df["source_file"] = os.path.basename(file_path)
    return df

In [21]:
sample_df = load_warn_fragment(table_files[0])
sample_df.head(10)

,0,1,2,3,4,5,6,source_file
0,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,WARN Report,Unnamed: 4,Unnamed: 5,warn7-1-20to6-3-21_table0.csv
1,NaN,NaN,NaN,NaN,Summary by State Fiscal Year,NaN,NaN,warn7-1-20to6-3-21_table0.csv
2,NaN,NaN,NaN,NaN,"July 1, 2020 - June 30, 2021",NaN,NaN,warn7-1-20to6-3-21_table0.csv
3,Notice,Effective,Received,NaN,NaN,NaN,No. Of,warn7-1-20to6-3-21_table0.csv
4,Date,Date,Date,Company,City,County,Employees Layoff/Closure Type,warn7-1-20to6-3-21_table0.csv
5,06/09/2020,06/07/2020,07/01/2020,Bay Club Santa Monica,Santa Monica,Los Angeles County,82 Layoff Permanent,warn7-1-20to6-3-21_table0.csv
6,06/19/2020,08/21/2020,07/01/2020,"Weber Metals, Inc",Paramount,Los Angeles County,169 Layoff Permanent,warn7-1-20to6-3-21_table0.csv
7,06/09/2020,06/07/2020,07/01/2020,Bay Club Rolling Hills,Rolling Hills Estates,Los Angeles County,64 Layoff Permanent,warn7-1-20to6-3-21_table0.csv
8,06/09/2020,06/07/2020,07/01/2020,Bay Club Redondo Beach,Redondo Beach,Los Angeles County,102 Layoff Permanent,warn7-1-20to6-3-21_table0.csv
9,06/09/2020,06/07/2020,07/01/2020,Bay Club Ross Valley,Kentfield,Marin County,51 Layoff Permanent,warn7-1-20to6-3-21_table0.csv


03: Standardize columns (fix “Unnamed: 0” headers)

Problem: Some fragments have real column names, others have Unnamed: 0, etc.
Plan: Rename columns by position into a consistent schema:

Target columns:
- notice_date
- effective_date
- received_date
- company
- city
- county
- employees
- layoff_closure_type

We will keep extra columns only if needed later.

In [75]:
# def standardize_warn_fragment(df: pd.DataFrame) -> pd.DataFrame:
#     """
#     Standardize one extracted WARN table fragment.

#     Handles two main layouts:

#     Layout A (older):
#     notice_date | effective_date | received_date | company | city | county | employees_type

#     Layout B (newer):
#     notice_date | effective_date | received_date | company | county | employees_type | address
#     """
#     df = df.copy()

#     # Keep first 7 columns + source_file
#     keep_cols = list(df.columns[:7]) + ["source_file"]
#     df = df[keep_cols]

#     # Temporary names
#     df.columns = [
#         "col0", "col1", "col2", "col3", "col4", "col5", "col6", "source_file"
#     ]

#     # Convert to strings
#     for c in ["col0", "col4", "col5", "col6"]:
#         df[c] = df[c].astype(str).str.strip()

#     # Only inspect rows that LOOK like real data rows
#     date_like = df["col0"].str.match(r"^\d{1,2}/\d{1,2}/\d{4}$", na=False)
#     sample = df.loc[date_like, ["col4", "col5", "col6"]].head(20).copy()

#     # Stronger layout detection using actual WARN keywords
#     col4_has_county = sample["col4"].str.contains("County", case=False, na=False).mean()
#     col5_has_county = sample["col5"].str.contains("County", case=False, na=False).mean()

#     col5_has_warn_type = sample["col5"].str.contains(
#         r"(Layoff|Closure|Relocation|Temporary|Permanent|Unknown|Not known)",
#         case=False,
#         na=False
#     ).mean()

#     col6_is_address = sample["col6"].str.contains(r"CA|Way|Blvd|Street|Rd", case=False, na=False).mean()

#     # Layout A: city | county | employees_type
#     if col5_has_county > 0.3 and col6_has_warn_type > 0.3:
#         df = df.rename(columns={
#             "col0": "notice_date",
#             "col1": "effective_date",
#             "col2": "received_date",
#             "col3": "company",
#             "col4": "city",
#             "col5": "county",
#             "col6": "employees_type"
#         })
#         df["address"] = pd.NA

#     # Layout B: county | employees_type | address
#     if col4_has_county > 0.3 and col5_starts_num > 0.3 and col6_is_address > 0.3:
#     # Standardize as Layout B
#         df.rename(columns={
#             "col0": "notice_date", "col1": "received_date", "col2": "effective_date",
#             "col3": "company", "col4": "county", "col5": "employees_type", "col6": "address"
#         }, inplace=True)
#         df["city"] = pd.NA

#     # Fallback: choose whichever column looks more like WARN type
#     elif col6_has_warn_type >= col5_has_warn_type:
#         df = df.rename(columns={
#             "col0": "notice_date",
#             "col1": "effective_date",
#             "col2": "received_date",
#             "col3": "company",
#             "col4": "city",
#             "col5": "county",
#             "col6": "employees_type"
#         })
#         df["address"] = pd.NA
#     else:
#         df = df.rename(columns={
#             "col0": "notice_date",
#             "col1": "effective_date",
#             "col2": "received_date",
#             "col3": "company",
#             "col4": "county",
#             "col5": "employees_type",
#             "col6": "address"
#         })
#         df["city"] = pd.NA

#     return df

In [91]:
def standardize_warn_fragment(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    num_cols = len(df.columns)
    temp_names = [f"col{i}" for i in range(min(num_cols, 8))]
    
    keep_cols = list(df.columns[:len(temp_names)])
    if "source_file" in df.columns:
        df = df[keep_cols + ["source_file"]]
        df.columns = temp_names + ["source_file"]
    else:
        df = df[keep_cols]
        df.columns = temp_names

    for c in [c for c in df.columns if c not in ["source_file"]]:
        df[c] = df[c].astype(str).str.strip()

    sample = df.dropna(subset=["col0"]).head(20)
    col4_has_county = sample["col4"].str.contains("County", case=False, na=False).mean()
    col5_has_county = sample["col5"].str.contains("County", case=False, na=False).mean()
    
    # Improved check: Employees columns are usually short (1-4 digits) 
    # while addresses are long and contain keywords like "Way" or "St"
    col5_is_address = sample["col5"].str.contains(r"Way|Blvd|Street|Rd|Ave", case=False, na=False).mean()
    col5_starts_num = sample["col5"].str.match(r"^\d", na=False).mean()

    # LAYOUT A: Older PDFs (Notice, Effective, Received, Company, City, County, Employees)
    if col5_has_county > 0.3:
        df = df.rename(columns={
            "col0": "notice_date", "col1": "effective_date", "col2": "received_date",
            "col3": "company", "col4": "city", "col5": "county", "col6": "employees_type"
        })
        df["address"] = pd.NA

    # LAYOUT B: Newer PDFs (Notice, Received, Effective, Company, County, Employees, Type, Address)
    elif col4_has_county > 0.3:
        df = df.rename(columns={
            "col0": "notice_date", 
            "col1": "received_date", # Swapped in 2024 
            "col2": "effective_date", # Swapped in 2024 
            "col3": "company", "col4": "county", "col5": "employees_type",
            "col6": "layoff_closure_type_raw", "col7": "address"
        })
        df["city"] = pd.NA
        if "col7" in df.columns:
            df["employees_type"] = df["employees_type"] + " " + df["layoff_closure_type_raw"].fillna("")
    
    else:
        # Fallback to standard layout 
        df = df.rename(columns={
            "col0": "notice_date", "col1": "effective_date", "col2": "received_date",
            "col3": "company", "col4": "city", "col5": "county", "col6": "employees_type"
        })

    return df

In [92]:
sample_std = standardize_warn_fragment(sample_df)
sample_std.head(10)

,notice_date,effective_date,received_date,company,city,county,employees_type,col7,source_file,address
0,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,WARN Report,Unnamed: 4,Unnamed: 5,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>
1,nan,nan,nan,nan,Summary by State Fiscal Year,nan,nan,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>
2,nan,nan,nan,nan,"July 1, 2020 - June 30, 2021",nan,nan,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>
3,Notice,Effective,Received,nan,nan,nan,No. Of,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>
4,Date,Date,Date,Company,City,County,Employees Layoff/Closure Type,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>
5,06/09/2020,06/07/2020,07/01/2020,Bay Club Santa Monica,Santa Monica,Los Angeles County,82 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>
6,06/19/2020,08/21/2020,07/01/2020,"Weber Metals, Inc",Paramount,Los Angeles County,169 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>
7,06/09/2020,06/07/2020,07/01/2020,Bay Club Rolling Hills,Rolling Hills Estates,Los Angeles County,64 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>
8,06/09/2020,06/07/2020,07/01/2020,Bay Club Redondo Beach,Redondo Beach,Los Angeles County,102 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>
9,06/09/2020,06/07/2020,07/01/2020,Bay Club Ross Valley,Kentfield,Marin County,51 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>


05: Split employees and layoff type

In the extracted WARN tables, the last useful field often looks like:
- 82 Layoff Permanent
- 97 Closure Permanent
- 71 Layoff Type Unknown

We split this into:
- employees
- layoff_closure_type

In [93]:
def clean_warn_fragment(df: pd.DataFrame) -> pd.DataFrame:
    df = standardize_warn_fragment(df)

    # Convert to string first and strip spaces
    df["notice_date"] = df["notice_date"].astype(str).str.strip()

    # Keep only rows that look like mm/dd/yyyy before parsing
    mask = df["notice_date"].str.match(r"^\d{1,2}/\d{1,2}/\d{4}$", na=False)
    df = df[mask].copy()

    # Parse dates
    df["notice_date_dt"] = pd.to_datetime(
        df["notice_date"],
        format="%m/%d/%Y",
        errors="coerce"
    )

    df = df.dropna(subset=["notice_date_dt"]).copy()

    return df

In [94]:
sample_clean = clean_warn_fragment(sample_df)
sample_clean.head(10)

,notice_date,effective_date,received_date,company,city,county,employees_type,col7,source_file,address,notice_date_dt
5,06/09/2020,06/07/2020,07/01/2020,Bay Club Santa Monica,Santa Monica,Los Angeles County,82 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>,2020-06-09
6,06/19/2020,08/21/2020,07/01/2020,"Weber Metals, Inc",Paramount,Los Angeles County,169 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>,2020-06-19
7,06/09/2020,06/07/2020,07/01/2020,Bay Club Rolling Hills,Rolling Hills Estates,Los Angeles County,64 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>,2020-06-09
8,06/09/2020,06/07/2020,07/01/2020,Bay Club Redondo Beach,Redondo Beach,Los Angeles County,102 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>,2020-06-09
9,06/09/2020,06/07/2020,07/01/2020,Bay Club Ross Valley,Kentfield,Marin County,51 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>,2020-06-09
10,06/09/2020,06/07/2020,07/01/2020,StoneTree Golf Club,Novato,Marin County,32 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>,2020-06-09
11,06/23/2020,06/23/2020,07/01/2020,The Freeman Company LLC,Anaheim,Orange County,29 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>,2020-06-23
12,06/18/2020,03/20/2020,07/01/2020,Hyatt Regency Sacramento,Sacramento,Sacramento County,203 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>,2020-06-18
13,06/18/2020,06/17/2020,07/01/2020,"Granite Summit, Inc",Running Springs,San Bernardino County,60 Layoff Type Unknown,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>,2020-06-18
14,06/18/2020,06/17/2020,07/01/2020,Pali Camp,Running Springs,San Bernardino County,71 Layoff Type Unknown,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>,2020-06-18


In [98]:
def split_employees_type(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    # 1. Capture numbers and commas at the START of the string only 
    df["employees"] = df["employees_type"].str.extract(r"^([\d,]+)", expand=False)
    
    # 2. Safety Check: If the extracted number is followed immediately by " " and a 
    # street-like word, it's likely an address, not a layoff count.
    is_address_mask = df["employees_type"].str.contains(r"^\d+\s+(Sierra|Hancock|Scripps|Champion|McCarthy|Lockheed)", case=False, na=False)
    df.loc[is_address_mask, "employees"] = np.nan
    
    # 3. Clean and convert 
    df["employees"] = df["employees"].str.replace(',', '', regex=True)
    df["employees"] = pd.to_numeric(df["employees"], errors="coerce")
    
    # 4. Extract layoff type 
    df["layoff_closure_type"] = df["employees_type"].str.replace(r"^\d+\s*", "", regex=True).str.strip()
    
    return df

In [99]:
sample_final = split_employees_type(sample_clean)
sample_final.head(10)

/var/folders/cm/87xs3xzs13qcmdnys2_jq0cc0000gn/T/ipykernel_74554/2714100290.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  is_address_mask = df["employees_type"].str.contains(r"^\d+\s+(Sierra|Hancock|Scripps|Champion|McCarthy|Lockheed)", case=False, na=False)


,notice_date,effective_date,received_date,company,city,county,employees_type,col7,source_file,address,notice_date_dt,employees,layoff_closure_type
5,06/09/2020,06/07/2020,07/01/2020,Bay Club Santa Monica,Santa Monica,Los Angeles County,82 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>,2020-06-09,82,Layoff Permanent
6,06/19/2020,08/21/2020,07/01/2020,"Weber Metals, Inc",Paramount,Los Angeles County,169 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>,2020-06-19,169,Layoff Permanent
7,06/09/2020,06/07/2020,07/01/2020,Bay Club Rolling Hills,Rolling Hills Estates,Los Angeles County,64 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>,2020-06-09,64,Layoff Permanent
8,06/09/2020,06/07/2020,07/01/2020,Bay Club Redondo Beach,Redondo Beach,Los Angeles County,102 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>,2020-06-09,102,Layoff Permanent
9,06/09/2020,06/07/2020,07/01/2020,Bay Club Ross Valley,Kentfield,Marin County,51 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>,2020-06-09,51,Layoff Permanent
10,06/09/2020,06/07/2020,07/01/2020,StoneTree Golf Club,Novato,Marin County,32 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>,2020-06-09,32,Layoff Permanent
11,06/23/2020,06/23/2020,07/01/2020,The Freeman Company LLC,Anaheim,Orange County,29 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>,2020-06-23,29,Layoff Permanent
12,06/18/2020,03/20/2020,07/01/2020,Hyatt Regency Sacramento,Sacramento,Sacramento County,203 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>,2020-06-18,203,Layoff Permanent
13,06/18/2020,06/17/2020,07/01/2020,"Granite Summit, Inc",Running Springs,San Bernardino County,60 Layoff Type Unknown,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>,2020-06-18,60,Layoff Type Unknown
14,06/18/2020,06/17/2020,07/01/2020,Pali Camp,Running Springs,San Bernardino County,71 Layoff Type Unknown,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>,2020-06-18,71,Layoff Type Unknown


In [102]:
sample_final.tail(10)

,notice_date,effective_date,received_date,company,city,county,employees_type,col7,source_file,address,notice_date_dt,employees,layoff_closure_type
10,06/09/2020,06/07/2020,07/01/2020,StoneTree Golf Club,Novato,Marin County,32 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>,2020-06-09,32,Layoff Permanent
11,06/23/2020,06/23/2020,07/01/2020,The Freeman Company LLC,Anaheim,Orange County,29 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>,2020-06-23,29,Layoff Permanent
12,06/18/2020,03/20/2020,07/01/2020,Hyatt Regency Sacramento,Sacramento,Sacramento County,203 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>,2020-06-18,203,Layoff Permanent
13,06/18/2020,06/17/2020,07/01/2020,"Granite Summit, Inc",Running Springs,San Bernardino County,60 Layoff Type Unknown,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>,2020-06-18,60,Layoff Type Unknown
14,06/18/2020,06/17/2020,07/01/2020,Pali Camp,Running Springs,San Bernardino County,71 Layoff Type Unknown,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>,2020-06-18,71,Layoff Type Unknown
15,06/18/2020,06/17/2020,07/01/2020,"Pali Institute, Inc",Running Springs,San Bernardino County,35 Layoff Type Unknown,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>,2020-06-18,35,Layoff Type Unknown
16,06/18/2020,06/17/2020,07/01/2020,"Pali Mountain Conference Center, Inc",Running Springs,San Bernardino County,17 Layoff Type Unknown,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>,2020-06-18,17,Layoff Type Unknown
17,06/09/2020,06/07/2020,07/01/2020,Bay Club San Francisco,San Francisco,San Francisco County,146 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>,2020-06-09,146,Layoff Permanent
18,06/09/2020,06/07/2020,07/01/2020,Bay Club SF Tennis,San Francisco,San Francisco County,64 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>,2020-06-09,64,Layoff Permanent
19,06/09/2020,06/07/2020,07/01/2020,Bay Club Redwood Shores,Redwood City,San Mateo County,298 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,<NA>,2020-06-09,298,Layoff Permanent


06: Process all extracted fragments

Now we apply the same cleaning pipeline to every extracted CSV fragment and combine them into one row-level WARN dataset.

In [103]:
cleaned_tables = []

for f in table_files:
    try:
        raw_df = load_warn_fragment(f)
        clean_df = clean_warn_fragment(raw_df)
        clean_df = split_employees_type(clean_df)
        cleaned_tables.append(clean_df)
    except Exception as e:
        print(f"Skipped {os.path.basename(f)} because of error: {e}")

len(cleaned_tables)

/var/folders/cm/87xs3xzs13qcmdnys2_jq0cc0000gn/T/ipykernel_74554/2714100290.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  is_address_mask = df["employees_type"].str.contains(r"^\d+\s+(Sierra|Hancock|Scripps|Champion|McCarthy|Lockheed)", case=False, na=False)
/var/folders/cm/87xs3xzs13qcmdnys2_jq0cc0000gn/T/ipykernel_74554/2714100290.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  is_address_mask = df["employees_type"].str.contains(r"^\d+\s+(Sierra|Hancock|Scripps|Champion|McCarthy|Lockheed)", case=False, na=False)
/var/folders/cm/87xs3xzs13qcmdnys2_jq0cc0000gn/T/ipykernel_74554/2714100290.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  is_address_mask = df["employees_type"].str.contains(r"^\d+\s+(Sierra|Hancoc

313

07: Combine all cleaned WARN fragments

At this point, each extracted CSV fragment has been:
- loaded safely with `header=None`
- standardized into a consistent column layout
- filtered to keep valid notice dates
- split into `employees` and `layoff_closure_type`

Now we combine all cleaned fragments into one row-level WARN dataset.

In [104]:
# Combine all cleaned tables into one dataframe
warn_clean_rows = pd.concat(cleaned_tables, ignore_index=True)

print("Combined shape:", warn_clean_rows.shape)
warn_clean_rows.head(10)

Combined shape: (7426, 14)


,notice_date,effective_date,received_date,company,city,county,employees_type,col7,source_file,address,notice_date_dt,employees,layoff_closure_type,layoff_closure_type_raw
0,06/09/2020,06/07/2020,07/01/2020,Bay Club Santa Monica,Santa Monica,Los Angeles County,82 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,NaN,2020-06-09,82.0,Layoff Permanent,NaN
1,06/19/2020,08/21/2020,07/01/2020,"Weber Metals, Inc",Paramount,Los Angeles County,169 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,NaN,2020-06-19,169.0,Layoff Permanent,NaN
2,06/09/2020,06/07/2020,07/01/2020,Bay Club Rolling Hills,Rolling Hills Estates,Los Angeles County,64 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,NaN,2020-06-09,64.0,Layoff Permanent,NaN
3,06/09/2020,06/07/2020,07/01/2020,Bay Club Redondo Beach,Redondo Beach,Los Angeles County,102 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,NaN,2020-06-09,102.0,Layoff Permanent,NaN
4,06/09/2020,06/07/2020,07/01/2020,Bay Club Ross Valley,Kentfield,Marin County,51 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,NaN,2020-06-09,51.0,Layoff Permanent,NaN
5,06/09/2020,06/07/2020,07/01/2020,StoneTree Golf Club,Novato,Marin County,32 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,NaN,2020-06-09,32.0,Layoff Permanent,NaN
6,06/23/2020,06/23/2020,07/01/2020,The Freeman Company LLC,Anaheim,Orange County,29 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,NaN,2020-06-23,29.0,Layoff Permanent,NaN
7,06/18/2020,03/20/2020,07/01/2020,Hyatt Regency Sacramento,Sacramento,Sacramento County,203 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,NaN,2020-06-18,203.0,Layoff Permanent,NaN
8,06/18/2020,06/17/2020,07/01/2020,"Granite Summit, Inc",Running Springs,San Bernardino County,60 Layoff Type Unknown,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,NaN,2020-06-18,60.0,Layoff Type Unknown,NaN
9,06/18/2020,06/17/2020,07/01/2020,Pali Camp,Running Springs,San Bernardino County,71 Layoff Type Unknown,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,NaN,2020-06-18,71.0,Layoff Type Unknown,NaN


## 08: Deduplicate repeated WARN rows

PDF extraction can duplicate rows across fragments or pages.
We remove duplicates using the main identifying columns.

In [105]:
before = len(warn_clean_rows)

warn_clean_rows = warn_clean_rows.drop_duplicates(
    subset=[
        "notice_date",
        "effective_date",
        "received_date",
        "company",
        "city",
        "county",
        "employees",
        "layoff_closure_type"
    ],
    keep="first"
)

after = len(warn_clean_rows)

print("Rows before dedup:", before)
print("Rows after dedup:", after)
print("Duplicates removed:", before - after)

Rows before dedup: 7426
Rows after dedup: 7142
Duplicates removed: 284


## 09: Save cleaned row-level WARN dataset

This file keeps one cleaned row per WARN notice and will be useful for debugging,
EDA, and future extensions.

**Output:**
- `data/processed/warn_clean_rows.csv`

In [106]:
os.makedirs("../data/processed", exist_ok=True)

warn_clean_out = warn_clean_rows.drop(columns=["notice_date_dt"]).copy()
warn_clean_out.to_csv("../data/processed/warn_clean_rows.csv", index=False)

print("Saved:", "../data/processed/warn_clean_rows.csv")
warn_clean_out.head()

Saved: ../data/processed/warn_clean_rows.csv


,notice_date,effective_date,received_date,company,city,county,employees_type,col7,source_file,address,employees,layoff_closure_type,layoff_closure_type_raw
0,06/09/2020,06/07/2020,07/01/2020,Bay Club Santa Monica,Santa Monica,Los Angeles County,82 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,NaN,82.0,Layoff Permanent,NaN
1,06/19/2020,08/21/2020,07/01/2020,"Weber Metals, Inc",Paramount,Los Angeles County,169 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,NaN,169.0,Layoff Permanent,NaN
2,06/09/2020,06/07/2020,07/01/2020,Bay Club Rolling Hills,Rolling Hills Estates,Los Angeles County,64 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,NaN,64.0,Layoff Permanent,NaN
3,06/09/2020,06/07/2020,07/01/2020,Bay Club Redondo Beach,Redondo Beach,Los Angeles County,102 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,NaN,102.0,Layoff Permanent,NaN
4,06/09/2020,06/07/2020,07/01/2020,Bay Club Ross Valley,Kentfield,Marin County,51 Layoff Permanent,warn7-1-20to6-3-21_table0.csv,warn7-1-20to6-3-21_table0.csv,NaN,51.0,Layoff Permanent,NaN


## 10: Aggregate monthly WARN layoffs (target variable)

For modeling, we use monthly WARN layoffs as the target variable.
We aggregate the total affected employees by month.

**Output:**
- `data/processed/warn_monthly.csv`

In [107]:
warn_clean_rows = warn_clean_rows[
    (warn_clean_rows["employees"].notna()) &
    (warn_clean_rows["employees"] > 0) &
    (warn_clean_rows["employees"] < 10000)
].copy()

In [108]:
warn_clean_rows.sort_values("employees", ascending=False)[
    ["notice_date", "company", "employees_type", "employees", "layoff_closure_type", "source_file"]
].head(20)

,notice_date,company,employees_type,employees,layoff_closure_type,source_file
5891,08/22/2023,U.S. Bank San Diego County,9918 Hibert Street San Diego CA 92131,9918.0,Hibert Street San Diego CA 92131,warn7-1-23-to6-30-24_table8.csv
5890,08/22/2023,U.S. Bank San Diego County,9918 Hibert Street San Diego CA 92131,9918.0,Hibert Street San Diego CA 92131,warn7-1-23-to6-30-24_table8.csv
5877,07/05/2023,U.S. Bank (1296) San Diego County,9885 Towne Centre Drive San Diego CA 92121,9885.0,Towne Centre Drive San Diego CA 92121,warn7-1-23-to6-30-24_table8.csv
5873,07/05/2023,U. S. Bank (1296) San Diego County,9885 Town Centre Drive San Diego CA 92121,9885.0,Town Centre Drive San Diego CA 92121,warn7-1-23-to6-30-24_table8.csv
5876,07/05/2023,U.S. Bank (1296) San Diego County,9885 Towne Centre Drive San Diego CA 92121,9885.0,Towne Centre Drive San Diego CA 92121,warn7-1-23-to6-30-24_table8.csv
5874,07/05/2023,U.S. Bank (1289) San Diego County,9865 Towne Centre Drive San Diego CA 92121,9865.0,Towne Centre Drive San Diego CA 92121,warn7-1-23-to6-30-24_table8.csv
5875,07/05/2023,U.S. Bank (1289) San Diego County,9865 Towne Centre Drive San Diego CA 92121,9865.0,Towne Centre Drive San Diego CA 92121,warn7-1-23-to6-30-24_table8.csv
5872,07/05/2023,U.S. Bank (1289) San Diego County,9865 Town Centre Drive San Diego CA 92121,9865.0,Town Centre Drive San Diego CA 92121,warn7-1-23-to6-30-24_table8.csv
6594,02/07/2025,AMERI-KLEEN dba Sanitation Specialists San Ben...,9351 Fairview Road Hollister CA 95023,9351.0,Fairview Road Hollister CA 95023,warn7-1-24to6-30-25_table30.csv
7219,05/23/2025,GTM Wholesale Liquidators dba GTM Discount Gen...,8967 Carlton Hills Blvd Santee CA 92071,8967.0,Carlton Hills Blvd Santee CA 92071,warn7-1-24to6-30-25_table50.csv


In [109]:
warn_clean_rows["month"] = warn_clean_rows["notice_date_dt"].dt.to_period("M").dt.to_timestamp()

warn_monthly = (
    warn_clean_rows.groupby("month", as_index=False)["employees"]
    .sum()
    .rename(columns={"employees": "warn_layoffs"})
    .sort_values("month")
)

warn_monthly.to_csv("../data/processed/warn_monthly.csv", index=False)

print("Saved:", "../data/processed/warn_monthly.csv")
warn_monthly.head(), warn_monthly.tail()

Saved: ../data/processed/warn_monthly.csv


(       month  warn_layoffs
 0 2019-11-01         330.0
 1 2020-01-01         250.0
 2 2020-03-01       12539.0
 3 2020-04-01       20584.0
 4 2020-05-01       12657.0,
         month  warn_layoffs
 61 2025-02-01      111727.0
 62 2025-03-01        5620.0
 63 2025-04-01        6484.0
 64 2025-05-01       35651.0
 65 2025-06-01       33701.0)